## Response Streaming

- To get live updates from LLM into user application


In [5]:
%pip install anthropic python-dotenv

# Load environment variables from .env file
from dotenv import load_dotenv

load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})

def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})

messages = []

add_user_message(messages, "What is the capital of France?")

streaming_response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in streaming_response:
    print(event)

RawMessageStartEvent(message=Message(id='msg_0118dYuQnne9QGtENQA3g2wE', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=14, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='The', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' capital of France is **Paris**.', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockStopEvent(index=0, type='content_block_stop')
RawMessageDeltaEvent(delta=Delta(container=None, stop_reason='end_turn', sto

**Types of Events streamed**

- `RawMessageStartEvent`: A new message is being sent
- `RawContentBlockStartEvent`: Start of a new block, which contains a text message, tool use, etc.
- `RawContentBlockDeltaEvent`: Chunk related to the latest content block that was started **contains actual response data**, so you get multiple of this.
- `RawContentBlockStopEvent`: Current content block has ended
- `RawMessageDeltaEvent`: Current message is complete
- `RawMessageStopEvent`: End of information about current message

In [13]:
# Getting the streamed response in a better way

messages = []

add_user_message(messages, "Write a 250 word essay about geography of Canada.")

with client.messages.stream(
    model=model,
    max_tokens=10000,
    messages=messages # no need for stream=True
) as stream:
    for text in stream.text_stream:
        print(text, end="")

# If you plan to store this in a backend database, and want to store the response as it is being streamed, you can do something like this:
stream.get_final_message()
    

# The Geography of Canada

Canada is the second largest country in the world by total area, covering approximately 9.98 million square kilometers. Stretching from the Atlantic Ocean in the east to the Pacific Ocean in the west, and northward to the Arctic Ocean, Canada's geography is remarkably diverse and vast.

The country is divided into several distinct geographical regions. The Canadian Shield, a massive plateau of ancient rock, dominates the central and eastern portions of the country. This rugged landscape covers nearly half of Canada's total landmass and contains thousands of lakes and rivers. The Great Plains, located in the prairie provinces of Alberta, Saskatchewan, and Manitoba, provide fertile agricultural land that produces significant quantities of wheat and other crops.

The Rocky Mountains rise dramatically in western Canada, forming a natural border between Alberta and British Columbia. These magnificent peaks attract tourists and outdoor enthusiasts from around the w

ParsedMessage(id='msg_019FMcdXDePKC8o1yPzmetVz', container=None, content=[ParsedTextBlock(citations=None, text="# The Geography of Canada\n\nCanada is the second largest country in the world by total area, covering approximately 9.98 million square kilometers. Stretching from the Atlantic Ocean in the east to the Pacific Ocean in the west, and northward to the Arctic Ocean, Canada's geography is remarkably diverse and vast.\n\nThe country is divided into several distinct geographical regions. The Canadian Shield, a massive plateau of ancient rock, dominates the central and eastern portions of the country. This rugged landscape covers nearly half of Canada's total landmass and contains thousands of lakes and rivers. The Great Plains, located in the prairie provinces of Alberta, Saskatchewan, and Manitoba, provide fertile agricultural land that produces significant quantities of wheat and other crops.\n\nThe Rocky Mountains rise dramatically in western Canada, forming a natural border be